## 数据导入

In [30]:
# 标签处理函数

import torch
from torch_geometric.loader import DataLoader
import pandas as pd
from incdbscan import IncrementalDBSCAN
import time

import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
import ast
import sys
import os
from get_base_pattern import get_base_pattern,base_pattern_parse
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


def parse_and_strip_benign(label_str):
    label_set = ast.literal_eval(label_str)  # 安全地解析字符串为集合
    new_set = label_set - {'BENIGN'}

    return str(new_set) if new_set else str(label_set)

## 一、数据的联合，embedding data, coo graph data, base pattern datadef get_base_pattern_data(coo_graph_df, embedding_data_df, encoded_signature_df):
def get_base_pattern_data(coo_graph_df, embedding_data_df):
    """
    该函数用于获取图的基本模式画像，并将其与嵌入数据和编码签名数据合并。

    参数:
        coo_graph_df (DataFrame): 包含图的COO格式数据的DataFrame。
        embedding_data_df (DataFrame): 包含嵌入数据的DataFrame。
    返回:
        DataFrame: 合并后的DataFrame，包含图的基本模式画像、嵌入数据和编码签名数据。
    """
    encoded_signature_path = "../..//dataset/match_knowledge/encoded_signature.csv"
    encoded_signature_df = pd.read_csv(encoded_signature_path)
    # 获取基本模式画像等图基本信息
    coo_graph_df['base_profile'] = ''
    coo_graph_df['base_pattern_parse'] = ''
    coo_graph_df['base_topologic'] = ''

    for i in range(len(coo_graph_df)):
        now_graph = coo_graph_df.iloc[i]
        # 获取基本模式画像
        base_pattern = get_base_pattern(now_graph)
        # 解析基本模式画像
        parse = base_pattern_parse(base_pattern, encoded_signature_df)
        # 中心节点个数，拓扑类型
        base_topologic = (base_pattern[0], base_pattern[3])
        # 将基本模式画像、解析结果和拓扑信息添加到DataFrame中
        coo_graph_df.loc[i, 'base_profile'] = str(base_pattern)
        coo_graph_df.loc[i, 'base_pattern_parse'] = str(parse)
        coo_graph_df.loc[i, 'base_topologic'] = str(base_topologic)

    # 合并COO图数据和嵌入数据
    graph_all_data = pd.merge(coo_graph_df, embedding_data_df, on=['id'], how='left')
    # 统计每个簇的告警数量
    edges_label = graph_all_data['edge_label'].to_list()
    warn_num_list = []
    for one_edge in edges_label:
        one_edge_label = eval(one_edge)
        warn_num = sum(one_edge_label.values())
        warn_num_list.append(warn_num)
    graph_all_data['warn_num'] = warn_num_list

    # 对label进行转换
    graph_all_data['group_label'] = graph_all_data['group_label'].apply(parse_and_strip_benign)
    condition = (graph_all_data['group_label'] == "{'BENIGN'}")

    # group_label_easy仅记录二分类结果
    graph_all_data['group_label_easy'] = "{'BENIGN'}"
    graph_all_data.loc[~condition,'group_label_easy'] = "{'High Risk'}"
    # risk_level记录二分类结果
    graph_all_data['risk_level'] = 0
    graph_all_data.loc[~condition,'risk_level'] = 1
    label_map = {
    "{'BENIGN'}": 0,
    "{'Bot'}": 1,
    "{'PortScan'}": 2,
    "{'DDoS'}": 3,
    "{'SSH-Patator'}": 4,
    "{'DoS Hulk'}": 5,
    "{'Web Attack ?Brute Force'}": 6,
    "{'Web Attack ?XSS'}": 7,
    "{'DoS slowloris'}": 8,
    "{'Infiltration'}": 9,
    "{'DoS Slowhttptest'}": 10,
    "{'DoS GoldenEye'}": 11,
    "{'FTP-Patator'}": 12,
    "{'Web Attack ?Sql Injection'}": 13
    }
    # label_id记录多分类结果
    graph_all_data['label_id'] = graph_all_data['group_label'].map(label_map)
    return graph_all_data


## 按照base画像进行粗分类
**基础base画像**：（中心节点个数，中心节点资产属性，边缘节点资产属性，拓扑类型，攻击类型，目的端口类型）  

如果按照base画像先分类后再聚类，则需要**base画像内**至少有一定数额的告警簇，才有进一步聚类的意义；  
如此就要求粗分类后，每个分类的中数量稍微多一些，即分类粗一些，目前考虑到的方法：

- **看看被筛掉的这些有什么特征**
- **划分攻击类型时候粗糙一些**
- **尝试改变时间窗的大小**
- 其他标签相同，攻击类型不同的，运行合并）


In [31]:
import pandas as pd
import numpy as np
import ast

# 全局超参数设定
coo_graph_train_path = "../..//dataset/graph-format-data/graph_train.csv"
train_embedding_path = "../..//src/evluation/graph_embeding_data/pre_and_tune/train/1/train_embedding.csv"
# 基础模式应包含的最小安全事件数量
min_base_pattern_graph_num = 1

coo_graph_df = pd.read_csv(coo_graph_train_path)
embedding_data_df = pd.read_csv(train_embedding_path)
embedding_data_df['embedding_data'] = embedding_data_df['embedding_data'].apply(ast.literal_eval)
graph_all_data = get_base_pattern_data(coo_graph_df,embedding_data_df)
graph_all_data['embedding_reduce'] = graph_all_data['embedding_data']

### 满足basic pattern基本数量,筛选graph data, 生成graph_all_data_select
pattern_classification = graph_all_data.groupby(['base_pattern_parse'])['id'].apply(list).reset_index()
pattern_classification = pattern_classification.rename(columns={'id': 'id_list'})
pattern_classification['count'] = pattern_classification['id_list'].apply(len)
limit_base_pattern = pattern_classification.loc[pattern_classification['count']>=min_base_pattern_graph_num]
graph_all_data_select = pd.merge(graph_all_data, limit_base_pattern, on=['base_pattern_parse'])
base_profile_support  = graph_all_data_select['base_pattern_parse'].unique()

In [32]:
graph_all_data['warn_num'].sum()

479887

In [33]:
count_no_noise_all, count_noise_all = 0, 0
cluster_distri_list = []
for one_base in base_profile_support:
    one_base_graph_data = graph_all_data_select[graph_all_data_select['base_pattern_parse'] == one_base]
    cluster_distri = one_base_graph_data.groupby(['group_label_easy']) \
    .agg({'id': 'count', 'warn_num': 'sum'}) \
    .reset_index() \
    .rename(columns={'id': 'graph_count'})
    cluster_distri['base_pattern_parse'] = one_base
    cluster_distri_list.append(cluster_distri)

# 使用pd.concat将所有DataFrame组合在一起
cluster_distri_res = pd.concat(cluster_distri_list, ignore_index=True)
cluster_distri_res = cluster_distri_res.rename(columns={'group_label_easy':'group_label'},)
cluster_distri_res

,group_label,graph_count,warn_num,base_pattern_parse
0,{'BENIGN'},2,878,"(1, ['DNS+DC Server'], 'Multi-type margin', 'd..."
1,{'BENIGN'},2,4,"(1, ['Outsider'], ['Ubuntu Client', 'MAC'], 'c..."
2,{'BENIGN'},75,75,"(1, ['DNS+DC Server'], ['Outsider'], ' simple ..."
3,{'BENIGN'},10,14,"(1, ['Outsider'], ['DNS+DC Server'], ' simple ..."
4,{'BENIGN'},135,197,"(1, ['MAC'], ['Outsider'], ' simple ', ['none'])"
...,...,...,...,...
1056,{'BENIGN'},1,17,"(1, ['Windows Client'], ['DNS+DC Server', 'Out..."
1057,{'BENIGN'},1,2,"(1, ['Ubuntu Client'], ['Outsider'], 'converge..."
1058,{'BENIGN'},1,5,"(1, ['Ubuntu Client'], ['Outsider'], 'converge..."
1059,{'High Risk'},2,134,"(1, ['Outsider'], ['Windows Client'], 'converg..."


In [34]:

from math import comb
def get_supsicous_prob(data:pd.DataFrame, k):
    """ 获取单个集群识别为supsicous的概率
    
    Args:
        data: 单个集群对应的数据表
        k: 采样个数
        
    Return:
        find_supsicous_prob: 识别该集群为supsicous的概率的概率
    """
    find_supsicous_prob = 0
    no_find_supsicous_prob = 0
    # 如果风险等级数量最多的个数小于k，则一定能识别为supsicous
    #print(data)
    group_label_max_num = data['graph_count'].max()
    graph_sum_num = data['graph_count'].sum()
    if group_label_max_num < k:
        find_supsicous_prob = 1
        return find_supsicous_prob
    # 否则在所有的风险等级分类中，找出个数>=k的，然后计算
    else:
        for index_count in range(len(data)):
            graph_count = data.loc[index_count,'graph_count']
            if graph_count >= k:
                no_find_supsicous_prob += comb(graph_count, k)/comb(graph_sum_num, k)
        find_supsicous_prob = 1 - no_find_supsicous_prob
        return find_supsicous_prob

def get_sample_num(unique_values_count:pd.DataFrame,clsuter_ok_distri,cluster_single_risk_percent):
    """ 获取单个集群的采样个数

    Args:
        data: 单个集群对应的数据表
        k: 采样个数

    Return:
        sample_num: 采样个数
    """
    for n_samples in range(1,100):
        supicious_clusters = unique_values_count[unique_values_count['group_label_num']>1]
        supicious_clusters = supicious_clusters.reset_index(drop=True)
        supicious_cluster_name = supicious_clusters['cluster_name'].unique()

        supsicous_prob_list = []
        for cluster_name in supicious_cluster_name:
            cluster_data = clsuter_ok_distri.loc[clsuter_ok_distri['cluster_name']==cluster_name]
            cluster_data = cluster_data.reset_index(drop=True)
            supsicous_prob = get_supsicous_prob(cluster_data,k=n_samples)
            supsicous_prob_list.append(supsicous_prob)
    
        supsicous_prob_mean = sum(supsicous_prob_list)/len(supsicous_prob_list)
        overall_supsicous_prob = supsicous_prob_mean * (1-cluster_single_risk_percent) + 1*cluster_single_risk_percent
        if overall_supsicous_prob>=0.96:
            return n_samples
    return {'error':'采样数量过大'}

# 手动分析阶段结果

In [35]:
### 手动阶段评估
# 统计噪声点（-1）的占比
clsuter_ok_distri = cluster_distri_res
clsuter_ok_distri['cluster_name'] = clsuter_ok_distri['base_pattern_parse']
# 覆盖率为100%

# 计算单一风险等级的占比
clsuter_ok_distri['cluster_name'] = clsuter_ok_distri['base_pattern_parse']
unique_values_count = clsuter_ok_distri.groupby(['cluster_name']) \
    .agg({'group_label': ['nunique', list], 'graph_count':'sum', 'warn_num':'sum'}) \
    .reset_index()
unique_values_count.columns = ['cluster_name', 'group_label_num', 'group_label_list', 'graph_count', 'warn_num']
one_single_risk_data = unique_values_count[unique_values_count['group_label_num']==1]
cluster_single_risk_num = len(one_single_risk_data)
cluster_single_risk_percent = cluster_single_risk_num / len(unique_values_count)
subgraph_single_risk_num = one_single_risk_data['graph_count'].sum()
subgraph_single_risk_percent = subgraph_single_risk_num / unique_values_count['graph_count'].sum()
alert_single_risk_num = one_single_risk_data['warn_num'].sum()
alert_single_risk_percent = alert_single_risk_num / unique_values_count['warn_num'].sum()
print('---------------------------------Workload reduction RESULTS----------------------------------------------------')
print("唯一风险等级集群占比: {:.2%}, 对应簇占比: {:.2%}, 告警占比{:.2%}".format(cluster_single_risk_percent,subgraph_single_risk_percent,alert_single_risk_percent))


# 计算采样数量&覆盖中的减少率、总的减少率
cluster_noise_num, subgraph_noise_num, alert_noise_num = 0, 0, 0
sample_num = get_sample_num(unique_values_count,clsuter_ok_distri,cluster_single_risk_percent)
print(f"采样数量为: {sample_num}")

grouped = clsuter_ok_distri.groupby(['base_pattern_parse'])
# 获取分组的数量
num_groups = grouped.ngroups
## 以告警事件簇为视角
# 计算覆盖率
no_select_subgraph = len(graph_all_data) -  len(graph_all_data_select)
subgraph_coverage = 1 - (no_select_subgraph + subgraph_noise_num) / len(graph_all_data)
# 覆盖中的减少率
cluster_hanlde_sugraph_num = sample_num * num_groups
subgraph_reduction = 1 - cluster_hanlde_sugraph_num / clsuter_ok_distri['warn_num'].sum()
# 总的减少率
total_subgraph_reduction = 1 - (no_select_subgraph + subgraph_noise_num + cluster_hanlde_sugraph_num) / len(graph_all_data)
total_alert_reduction = 1 - (no_select_subgraph + subgraph_noise_num + cluster_hanlde_sugraph_num) / graph_all_data['warn_num'].sum()
print("告警事件簇覆盖率为: {:.2%}, 覆盖部分的减少率为: {:.2%}, 总的减少率为{:.2%}，告警日志的总减少率为{:.2%}".format(subgraph_coverage,subgraph_reduction,total_subgraph_reduction,total_alert_reduction))


---------------------------------Workload reduction RESULTS----------------------------------------------------
唯一风险等级集群占比: 98.76%, 对应簇占比: 91.04%, 告警占比40.49%
采样数量为: 1
告警事件簇覆盖率为: 100.00%, 覆盖部分的减少率为: 99.78%, 总的减少率为88.27%，告警日志的总减少率为99.78%


# 自动化处置阶段结果

In [36]:
# 读入测试数据
### 自动阶段评估
coo_graph_test_path = "../..//dataset/graph-format-data/graph_test.csv"
test_embedding_path = "../..//src/evluation/graph_embeding_data/pre_and_tune/test/1/test_embedding.csv"
coo_test_df = pd.read_csv(coo_graph_test_path)
embedding_test_df = pd.read_csv(test_embedding_path)
embedding_test_df['embedding_data'] = embedding_test_df['embedding_data'].apply(ast.literal_eval)
graph_test_data = get_base_pattern_data(coo_test_df,embedding_test_df)

In [37]:
risk_mapping = {
    'BENIGN': 0,
    'High Risk': 1
}
cluster_distri_res['group_label'] = cluster_distri_res['group_label'].apply(lambda x: [risk_mapping[label] for label in eval(x)])
match_df = cluster_distri_res.groupby('base_pattern_parse')['group_label'].agg(list).reset_index()
# 定义一个函数来取列表中的最大值
def max_value(lst):
    return max(max(sublst) for sublst in lst)

# 应用max_value函数取每个列表的最大值
match_df['group_label'] = match_df['group_label'].apply(max_value)
match_df = match_df.rename(columns={'group_label': 'match_label'})
succeed_match_data = pd.merge(graph_test_data, match_df, on='base_pattern_parse', how='inner')
succeed_match_prob = len(succeed_match_data) / len(graph_test_data)
succeed_log_match_prob = succeed_match_data['warn_num'].sum() / graph_test_data['warn_num'].sum()

print('---------------------------------RESULTS----------------------------------------------------')
print('events匹配成功率为：{:.2%}，日志匹配成功率为：{:.2%}'.format(succeed_match_prob,succeed_log_match_prob))

# 匹配准确率，召回率等结果
from sklearn.metrics import classification_report

report = classification_report(succeed_match_data['risk_level'], succeed_match_data['match_label'], sample_weight=succeed_match_data['warn_num'],output_dict=True)
for class_label, metrics in report.items():
    print(f"Class {class_label}:")
    
    # Check if metrics is a dictionary or a float
    if isinstance(metrics, dict):
        print("  Precision: {:.2%}".format(metrics['precision']))
        print("  Recall: {:.2%}".format(metrics['recall']))
        print("  F1 Score: {:.2%}".format(metrics['f1-score']))
        print("  Support: {:n}".format(metrics['support']))
    else:
        print(f"  {metrics}")
# 计算低估风向比例(succeed_match['risk_level'] > succeed_match['match_risk']).sum()
total_low_estimation_rate = succeed_match_data[(succeed_match_data['risk_level'] > succeed_match_data['match_label'])]['warn_num'].sum()/succeed_match_data['warn_num'].sum()

# 计算每个分类的低估率
low_estimation_rates = {}
for i in range(0, 2):
    mask = (succeed_match_data['risk_level'] == i) & (succeed_match_data['match_label'] < i)
    count = succeed_match_data[mask]['warn_num'].sum()
    print(i,count)
    rate = count/succeed_match_data.loc[succeed_match_data['risk_level'] == i]['warn_num'].sum()
    low_estimation_rates[i] = rate

print("整体低估率:{:.4%}".format(total_low_estimation_rate))
print("分类低估率:", low_estimation_rates)

---------------------------------RESULTS----------------------------------------------------
events匹配成功率为：91.01%，日志匹配成功率为：92.53%
Class 0:
  Precision: 99.44%
  Recall: 62.20%
  F1 Score: 76.53%
  Support: 48976
Class 1:
  Precision: 88.41%
  Recall: 99.88%
  F1 Score: 93.79%
  Support: 141330
Class accuracy:
  0.9018160226162076
Class macro avg:
  Precision: 93.92%
  Recall: 81.04%
  F1 Score: 85.16%
  Support: 190306
Class weighted avg:
  Precision: 91.25%
  Recall: 90.18%
  F1 Score: 89.35%
  Support: 190306
0 0
1 171
整体低估率:0.0899%
分类低估率: {0: 0.0, 1: 0.0012099341965612397}


获取按照[one_base_pattern, HDBSCAN_LABEL, group_label] 的分布，以及**噪声点**个数的占比

In [ ]:
from incdbscan import IncrementalDBSCAN

base_profile_support  = graph_all_data_select['base_pattern_parse'].unique()
count_no_noise_all, count_noise_all = 0, 0
cluster_distri_list = []

# 增量聚类
base_dbscan = {}
base_graphId = {}
base_embedding = {}
base_risk = {}
for i,one_base in enumerate(base_profile_support):
    one_base_graph_data = graph_all_data_select[graph_all_data_select['base_pattern_parse'] == one_base]
    base_dbscan[one_base] = IncrementalDBSCAN(eps=0.07, min_pts=3)
    cluster_distri, count_no_noise, count_noise = one_base_cluster(one_base_graph_data,base_dbscan[one_base])
    cluster_distri_list.append(cluster_distri)
    count_no_noise_all += count_no_noise
    count_noise_all += count_noise
    
# 使用pd.concat将所有DataFrame组合在一起
cluster_distri_res = pd.concat(cluster_distri_list, ignore_index=True)

print(f"count_no_noise_all = {count_no_noise_all}, count_noise_all = {count_noise_all}")
cluster_distri_res